# PulseInsight DuckDB Notebook

This notebook sets up the DuckDB database for the PulseInsight project and runs the core analytical queries used in the report and demo.

## 1. Setup
This notebook connects to the local DuckDB database file and loads the diabetes CSV dataset into a table if needed.

In [1]:
import duckdb

db_path = r"/Users/elliehe/Desktop/DSCI_551/Project/diabetes.db"
csv_path = r"/Users/elliehe/Desktop/DSCI_551/Project/diabetes.csv"

con = duckdb.connect(db_path)
con.execute("DROP TABLE IF EXISTS diabetes")
con.execute(f"""
CREATE TABLE diabetes AS
SELECT *
FROM read_csv_auto('{csv_path}');
""")

print("Connected to:", db_path)
print("Loaded dataset from:", csv_path)

Connected to: /Users/elliehe/Desktop/DSCI_551/Project/diabetes.db
Loaded dataset from: /Users/elliehe/Desktop/DSCI_551/Project/diabetes.csv


## 2. Verify Table and Schema

In [2]:
con.execute("SHOW TABLES").fetchdf()

,name
0,diabetes


In [3]:
con.execute("SELECT COUNT(*) FROM diabetes").fetchdf()

,count_star()
0,768


In [4]:
con.execute("DESCRIBE diabetes").fetchdf()

,column_name,column_type,null,key,default,extra
0,Pregnancies,BIGINT,YES,None,None,None
1,Glucose,BIGINT,YES,None,None,None
2,BloodPressure,BIGINT,YES,None,None,None
3,SkinThickness,BIGINT,YES,None,None,None
4,Insulin,BIGINT,YES,None,None,None
5,BMI,DOUBLE,YES,None,None,None
6,DiabetesPedigreeFunction,DOUBLE,YES,None,None,None
7,Age,BIGINT,YES,None,None,None
8,Outcome,BIGINT,YES,None,None,None


## 3. Preview the Data

In [5]:
con.execute("SELECT * FROM diabetes LIMIT 5").fetchdf()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 4. Core Analytical Queries

### Query 1: Average Glucose

In [6]:
con.execute("SELECT AVG(Glucose) FROM diabetes").fetchdf()

,avg(Glucose)
0,120.894531


### Query 2: Average BMI Grouped by Outcome

In [7]:
con.execute("SELECT Outcome, AVG(BMI) FROM diabetes GROUP BY Outcome").fetchdf()

,Outcome,avg(BMI)
0,0,30.304200
1,1,35.142537


### Query 3: Age Range Query

In [8]:
con.execute("SELECT * FROM diabetes WHERE Age BETWEEN 30 AND 50").fetchdf()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,0,137,40,35,168,43.1,2.288,33,1
4,5,116,74,0,0,25.6,0.201,30,0
...,...,...,...,...,...,...,...,...,...
286,7,137,90,41,0,32.0,0.391,39,0
287,9,170,74,31,0,44.0,0.403,43,1
288,9,89,62,0,0,22.5,0.142,33,0
289,5,121,72,23,112,26.2,0.245,30,0


### Query 4: High-Risk Patient Filtering

In [9]:
con.execute("SELECT * FROM diabetes WHERE Glucose > 150 AND BMI > 35").fetchdf()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,10,168,74,0,0,38.0,0.537,34,1
1,7,196,90,0,0,39.8,0.451,41,1
2,9,171,110,24,240,45.4,0.721,54,1
3,0,180,66,39,0,42.0,1.893,25,1
4,7,187,68,39,304,37.7,0.254,41,1
...,...,...,...,...,...,...,...,...,...
58,13,153,88,37,140,40.6,1.174,39,0
59,3,187,70,22,200,36.4,0.408,36,1
60,0,181,88,44,510,43.3,0.222,26,1
61,6,190,92,0,0,35.5,0.278,66,1


## 5. Optional Demo Queries
These are extra queries that may be useful in the project demo or final report.

In [10]:
demo_queries = {
    "Average glucose": "SELECT AVG(Glucose) FROM diabetes;",
    "High BMI patients": "SELECT * FROM diabetes WHERE BMI > 35;",
    "Outcome group comparison": "SELECT Outcome, AVG(Glucose) FROM diabetes GROUP BY Outcome;",
    "Age cohort": "SELECT * FROM diabetes WHERE Age BETWEEN 30 AND 50;"
}
demo_queries

{'Average glucose': 'SELECT AVG(Glucose) FROM diabetes;',
 'High BMI patients': 'SELECT * FROM diabetes WHERE BMI > 35;',
 'Outcome group comparison': 'SELECT Outcome, AVG(Glucose) FROM diabetes GROUP BY Outcome;',
 'Age cohort': 'SELECT * FROM diabetes WHERE Age BETWEEN 30 AND 50;'}

## 6. Close Connection

In [11]:
con.close()